# 📊 Exploración de Datos: Sistema de Calificaciones Escolar

> **Nota de Arquitectura**: Este notebook es un entregable de **exploración preliminar (EDA)** para entender el esquema, distribuciones y anomalías de los datos generados. **No es parte del pipeline de producción**, el cual se ejecuta mediante scripts modulares y orquestación Airflow.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# Rutas relativas
DATA_RAW = Path('../data/raw')

courses_df = pd.read_csv(DATA_RAW / 'courses.csv')
subjects_df = pd.read_csv(DATA_RAW / 'subjects.csv')
students_df = pd.read_csv(DATA_RAW / 'students.csv')
assessments_df = pd.read_csv(DATA_RAW / 'assessments.csv')
grades_df = pd.read_csv(DATA_RAW / 'grades.csv')

print(f"Cursos cargados: {len(courses_df)}")
print(f"Materias cargadas: {len(subjects_df)}")
print(f"Estudiantes cargados: {len(students_df)}")
print(f"Evaluaciones cargadas: {len(assessments_df)}")
print(f"Notas cargadas: {len(grades_df)}")

## 1. Inspección de Cursos y Materias

In [ ]:
display(courses_df)
display(subjects_df.head(10))

## 2. Ponderación de Evaluaciones por Materia
Regla de negocio: La suma de porcentajes de las evaluaciones de cada materia debe ser exactamente **100%**.

In [ ]:
# Verificación de suma de pesos por materia
weights_summary = assessments_df.groupby('subject_id')['weight_percentage'].sum().reset_index()
weights_summary.columns = ['subject_id', 'total_weight_percentage']

invalid_weights = weights_summary[weights_summary['total_weight_percentage'] != 100.0]
print(f"Materias con peso != 100%: {len(invalid_weights)}")
display(invalid_weights)

## 3. Distribución y Calidad de Calificaciones (Grades)
Escala de evaluación: 0.0 a 5.0 (Aprobación >= 3.0).

In [ ]:
display(grades_df['score'].describe())

# Identificación de anomalías intencionales en raw
out_of_range = grades_df[(grades_df['score'] < 0.0) | (grades_df['score'] > 5.0)]
null_scores = grades_df[grades_df['score'].isnull()]
duplicate_ids = grades_df[grades_df['grade_id'].duplicated(keep=False)]

print(f"Notas fuera de rango [0.0 - 5.0]: {len(out_of_range)}")
print(f"Notas con valor nulo: {len(null_scores)}")
print(f"Registros con grade_id duplicado: {len(duplicate_ids)}")

display(out_of_range)

## 4. Conclusiones del Diagnóstico Exploratorio
- Los datos raw contienen casos de prueba de calidad reales (nulos, valores atípicos y duplicados).
- La regla de validación debe filtrar los registros corruptos y enviarlos a `data/errors/` con su motivo.
- Los datos limpios continuarán hacia la ingesta en PostgreSQL (`raw` -> `staging` -> `silver` -> `gold`).